In [ ]:
import numpy as np
import re
import gfapy
import networkx as nx
import argparse

from itertools import combinations

from qiskit import transpile
from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import QAOAAnsatz


from sympy import Poly, Symbol

from qiskit_aer import AerSimulator
from qiskit_ibm_runtime.fake_provider import FakeFez

from pysat.formula import CNF, IDPool
from pysat.solvers import Solver
from qiskit.transpiler.passes.routing.commuting_2q_gate_routing import SwapStrategy


In [2]:
class Binary(Symbol):
    def _eval_power(self, other):
        return self
    
    
def monomial_to_pauli(monomial):
    indices = [int(re.search(r'[0-9]+', atom.name).group(0)) for atom in monomial.atoms()]
    pauli_str = ['I'] * n * T
    for i in indices:
        pauli_str[i] = 'Z'
    return ''.join(pauli_str)


def two_qubit_count(qc):
    return qc.count_ops().get("cz", 0) + qc.count_ops().get("rzz", 0) + qc.count_ops().get("cx", 0)


def depth(qc):
    return qc.depth(lambda instr: len(instr.qubits) > 1)


def bin_rep(k):
    return [int(x) for x in np.binary_repr(k, n)[::-1]]

In [3]:
fake_fez = FakeFez()

backend = AerSimulator.from_backend(fake_fez)

In [4]:
filename = 'test_N3_W4'
filepath = f'/nfs/users/nfs_j/jc59/quantumwork/pangenome/data/{filename}.gfa'

gfa = gfapy.Gfa.from_file(filepath, vlevel=0)

In [5]:
copy_numbers = [2,1,1]

In [6]:
graph = nx.DiGraph()
for index, segment_line in enumerate(gfa.segments):
    graph.add_node(f'{segment_line.name}_+', weight=copy_numbers[index], start=segment_line.st)
    graph.add_node(f'{segment_line.name}_-', weight=copy_numbers[index], start=segment_line.st)
for edge_line in gfa.edges:
    v1 = edge_line.sid1
    v2 = edge_line.sid2
    graph.add_edges_from([
        (f'{v1.name}_{v1.orient}', f'{v2.name}_{v2.orient}'),
    ])
    v1.invert()
    v2.invert()
    graph.add_edges_from([
        (f'{v2.name}_{v2.orient}', f'{v1.name}_{v1.orient}'),
    ])

nodes = list(graph.nodes)
N = len(nodes)
n = int(np.ceil(np.log2(N+1)))
total_weight = int(sum(graph.nodes[node]["weight"] for node in nodes) / 2)
T = int(1.1 * total_weight)

x = [[Binary(f'x[{t}][{i}]') for i in range(n)] for t in range(T)]

In [7]:
constraint = sum([
    1 - sum([
        np.prod([
            1 - x[t][k] - bin_rep(i)[k] + 2 * x[t][k] * bin_rep(i)[k]
        for k in range(n)]) * sum([
            np.prod([
                1 - x[t+1][k] - bin_rep(j)[k] + 2 * x[t+1][k] * bin_rep(j)[k]
            for k in range(n)])
        for j in [nodes.index(nbr) for nbr in graph.neighbors(nodes[i])]])
    for i in range(N)])
for t in range(T-1)])

obj = sum([
    (
        sum([
            np.prod([
                1 - x[t][k] - bin_rep(i)[k] + 2 * x[t][k] * bin_rep(i)[k]
            for k in range(n)])
            + np.prod([
                1 - x[t][k] - bin_rep(i+1)[k] + 2 * x[t][k] * bin_rep(i+1)[k]
            for k in range(n)])
        for t in range(T)])
        - graph.nodes[nodes[i]]["weight"]
    ) ** 2
for i in range(0,N,2)])


lamda = 10
total = lamda * constraint + obj

Z = [Binary(f"Z[{i}]") for i in range(n*T)]
ising = total.subs(zip([item for row in x for item in row], [0.5 - z/2 for z in Z]))

ising = Poly(ising, Z)

ising_expr_coeffs = ising.as_expr().as_coefficients_dict()


In [8]:
hamiltonian = SparsePauliOp('I'*n*T, ising_expr_coeffs[1])
for (monomial, coeff) in ising_expr_coeffs.items():
    if monomial == 1:
        continue
    hamiltonian += SparsePauliOp(monomial_to_pauli(monomial), coeff)
hamiltonian = hamiltonian.sort(weight=True)

In [142]:
edges = []
weights = []
for t in hamiltonian:
    t.coeffs[0]
    if np.sum(t.paulis[0].z) == 2:
        edge = np.nonzero(t.paulis[0].z)[0]
        edges.append(edge)
        weights.append(t.coeffs[0])
program_graph = nx.Graph()
for idx in range(len(weights)):
    program_graph.add_edge(edges[idx][0],edges[idx][1],weight=weights[idx])

In [143]:
program_graph.edges.data()

EdgeDataView([(np.int64(0), np.int64(1), {'weight': np.complex128(0.125+0j)}), (np.int64(0), np.int64(3), {'weight': np.complex128(0.6875+0j)}), (np.int64(0), np.int64(4), {'weight': np.complex128(-0.4375+0j)}), (np.int64(0), np.int64(6), {'weight': np.complex128(0.375+0j)}), (np.int64(0), np.int64(7), {'weight': np.complex128(-0.125+0j)}), (np.int64(0), np.int64(9), {'weight': np.complex128(0.375+0j)}), (np.int64(0), np.int64(10), {'weight': np.complex128(-0.125+0j)}), (np.int64(1), np.int64(3), {'weight': np.complex128(-0.4375+0j)}), (np.int64(1), np.int64(4), {'weight': np.complex128(0.6875+0j)}), (np.int64(1), np.int64(5), {'weight': np.complex128(0.625+0j)}), (np.int64(1), np.int64(6), {'weight': np.complex128(-0.125+0j)}), (np.int64(1), np.int64(7), {'weight': np.complex128(0.375+0j)}), (np.int64(1), np.int64(9), {'weight': np.complex128(-0.125+0j)}), (np.int64(1), np.int64(10), {'weight': np.complex128(0.375+0j)}), (np.int64(3), np.int64(4), {'weight': np.complex128(0.4375+0j)})

In [66]:
num_nodes_g1 = 6 # T*n
swap_strategy = SwapStrategy.from_line(list(range(6)))

num_nodes_g2 = swap_strategy.distance_matrix.shape[0]

In [147]:
e = [(0,0),(0,1),(1,2),(2,3)]
program_graph = nx.Graph(e)


In [148]:
min_layers = 0
max_layers = num_nodes_g1 -1

variable_pool = IDPool(start_from=1)
variables = np.array(
    [
        [variable_pool.id(f"v_{i}_{j}") for j in range(num_nodes_g2)]
        for i in range(num_nodes_g1)
    ],
    dtype=int,
)
vid2mapping = {v: idx for idx, v in np.ndenumerate(variables)}
binary_search_results = {}

def interrupt(solver):
    # This function is called to interrupt the solver when the timeout is reached.
    solver.interrupt()

# Make a cnf for the one-to-one mapping constraint
cnf1 = []
for i in range(num_nodes_g1):
    clause = variables[i, :].tolist()
    cnf1.append(clause)
    for k, m in combinations(clause, 2):
        cnf1.append([-1 * k, -1 * m])
for j in range(num_nodes_g2):
    clause = variables[:, j].tolist()
    for k, m in combinations(clause, 2):
        cnf1.append([-1 * k, -1 * m])

# # Perform a binary search over the number of swap layers to find the minimum
# # number of swap layers that satisfies the subgraph isomorphism problem.
# while min_layers < max_layers:
num_layers = (min_layers + max_layers) // 2

# Create the connectivity matrix. Note that if the swap strategy cannot reach
# full connectivity then its distance matrix will have entries with -1. These
# entries must be treated as False.
d_matrix = swap_strategy.distance_matrix
connectivity_matrix = ((-1 < d_matrix) & (d_matrix <= num_layers)).astype(int)
# Make a cnf for the adjacency constraint
cnf2 = []
for e_0, e_1 in program_graph.edges:
    clause_matrix = np.multiply(connectivity_matrix, variables[e_1, :])
    clause = np.concatenate(
        (
            [[-variables[e_0, i]] for i in range(num_nodes_g2)],
            clause_matrix,
        ),
        axis=1,
    )
    # Remove 0s from each clause
    cnf2.extend([c[c != 0].tolist() for c in clause])

cnf = CNF(from_clauses=cnf1 + cnf2)

In [149]:
swap_strategy

SwapStrategy with swap layers:
((0, 1), (2, 3), (4, 5)),
((1, 2), (3, 4)),
((0, 1), (2, 3), (4, 5)),
((1, 2), (3, 4)),
on [[0, 1], [1, 0], [1, 2], [2, 1], [2, 3], [3, 2], [3, 4], [4, 3], [4, 5], [5, 4]] coupling map.

In [150]:
d_matrix # Entry [i,j] gives the first layer at which qubits i,j neighbour

array([[0, 0, 4, 1, 3, 2],
       [0, 0, 0, 2, 4, 3],
       [4, 0, 0, 0, 2, 1],
       [1, 2, 0, 0, 0, 4],
       [3, 4, 2, 0, 0, 0],
       [2, 3, 1, 4, 0, 0]])

In [71]:
connectivity_matrix # Entry [i,j] gives whether [i,j] neighbour within num_layer layers of swaps

array([[1, 1, 0, 1, 0, 1],
       [1, 1, 1, 1, 0, 0],
       [0, 1, 1, 1, 1, 1],
       [1, 1, 1, 1, 1, 0],
       [0, 0, 1, 1, 1, 1],
       [1, 0, 1, 0, 1, 1]])

In [31]:
cnf2 = []
for e_0, e_1 in program_graph.edges:
    print(e_0, e_1)
    print(variables[e_1, :])
    clause_matrix = np.multiply(connectivity_matrix, variables[e_1, :])
    
    print(clause_matrix)
    clause = np.concatenate(
        (
            [[-variables[e_0, i]] for i in range(num_nodes_g2)],
            clause_matrix,
        ),
        axis=1,
    )
    print(clause)
    for c in clause:
        print(c[c != 0].tolist())
    # Remove 0s from each clause
    cnf2.extend([c[c != 0].tolist() for c in clause])
    print('')

0 0
[1 2 3 4]
[[1 2 0 4]
 [1 2 3 0]
 [0 2 3 4]
 [1 0 3 4]]
[[-1  1  2  0  4]
 [-2  1  2  3  0]
 [-3  0  2  3  4]
 [-4  1  0  3  4]]
[-1, 1, 2, 4]
[-2, 1, 2, 3]
[-3, 2, 3, 4]
[-4, 1, 3, 4]

0 1
[5 6 7 8]
[[5 6 0 8]
 [5 6 7 0]
 [0 6 7 8]
 [5 0 7 8]]
[[-1  5  6  0  8]
 [-2  5  6  7  0]
 [-3  0  6  7  8]
 [-4  5  0  7  8]]
[-1, 5, 6, 8]
[-2, 5, 6, 7]
[-3, 6, 7, 8]
[-4, 5, 7, 8]

1 2
[ 9 10 11 12]
[[ 9 10  0 12]
 [ 9 10 11  0]
 [ 0 10 11 12]
 [ 9  0 11 12]]
[[-5  9 10  0 12]
 [-6  9 10 11  0]
 [-7  0 10 11 12]
 [-8  9  0 11 12]]
[-5, 9, 10, 12]
[-6, 9, 10, 11]
[-7, 10, 11, 12]
[-8, 9, 11, 12]

2 3
[13 14 15 16]
[[13 14  0 16]
 [13 14 15  0]
 [ 0 14 15 16]
 [13  0 15 16]]
[[ -9  13  14   0  16]
 [-10  13  14  15   0]
 [-11   0  14  15  16]
 [-12  13   0  15  16]]
[-9, 13, 14, 16]
[-10, 13, 14, 15]
[-11, 14, 15, 16]
[-12, 13, 15, 16]



In [ ]:
# [-1, 5,6,8] means: if 1 (i.e. if qubit 0 maps to 0), then either [5,6 or 8] ie not 7 ie not qubit 1 maps to 2, which is 0s "blind spot"
# Say we have edge (0,1,2)
# Need to track connecitivty of triples?
# e.g. d_tensor[i,j,k] = first layer when [i,j,k] are all neighbouring (any order)
# connectivity_tensor[i,j,k] = whether [i,j,k] all neighbour within num_layers swaps

# 0,1,2,3,4,5
# After 1 swap
# 1,0,3,2,5,4
# [(0,1,2),(1,2,3),(2,3,4),(3,4,5),(1,0,3),(0,3,2),(3,2,5),(2,5,4)]

In [95]:
couples = list(swap_strategy.swapped_coupling_map(1))

In [96]:
from itertools import permutations

In [111]:
num_qubits = 3
perms = []

for i in range(num_layers+1):
    couples = list(swap_strategy.swapped_coupling_map(i))
    for perm in permutations(range(num_nodes_g2), num_qubits):
        if (perm[0],perm[1]) in couples and (perm[1],perm[2]) in couples:
            perms.append((perm[0],perm[1], perm[2]))
perms

[(0, 1, 2),
 (1, 2, 3),
 (2, 1, 0),
 (2, 3, 4),
 (3, 2, 1),
 (3, 4, 5),
 (4, 3, 2),
 (5, 4, 3),
 (0, 3, 2),
 (1, 0, 3),
 (2, 3, 0),
 (2, 5, 4),
 (3, 0, 1),
 (3, 2, 5),
 (4, 5, 2),
 (5, 2, 3),
 (0, 3, 1),
 (0, 5, 2),
 (1, 3, 0),
 (2, 5, 0),
 (3, 0, 5),
 (4, 2, 5),
 (5, 0, 3),
 (5, 2, 4)]

In [117]:
connectivity_tensor = np.zeros((num_nodes_g2,num_nodes_g2,num_nodes_g2))
for perm in perms:
    for rotn in permutations(perm, len(perm)):
        connectivity_tensor[rotn] = 1
connectivity_tensor = (connectivity_tensor).astype(int)
connectivity_tensor

array([[[0, 0, 0, 0, 0, 0],
        [0, 0, 1, 1, 0, 0],
        [0, 1, 0, 1, 0, 1],
        [0, 1, 1, 0, 0, 1],
        [0, 0, 0, 0, 0, 0],
        [0, 0, 1, 1, 0, 0]],

       [[0, 0, 1, 1, 0, 0],
        [0, 0, 0, 0, 0, 0],
        [1, 0, 0, 1, 0, 0],
        [1, 0, 1, 0, 0, 0],
        [0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0]],

       [[0, 1, 0, 1, 0, 1],
        [1, 0, 0, 1, 0, 0],
        [0, 0, 0, 0, 0, 0],
        [1, 1, 0, 0, 1, 1],
        [0, 0, 0, 1, 0, 1],
        [1, 0, 0, 1, 1, 0]],

       [[0, 1, 1, 0, 0, 1],
        [1, 0, 1, 0, 0, 0],
        [1, 1, 0, 0, 1, 1],
        [0, 0, 0, 0, 0, 0],
        [0, 0, 1, 0, 0, 1],
        [1, 0, 1, 0, 1, 0]],

       [[0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0],
        [0, 0, 0, 1, 0, 1],
        [0, 0, 1, 0, 0, 1],
        [0, 0, 0, 0, 0, 0],
        [0, 0, 1, 1, 0, 0]],

       [[0, 0, 1, 1, 0, 0],
        [0, 0, 0, 0, 0, 0],
        [1, 0, 0, 1, 1, 0],
        [1, 0, 1, 0, 1, 0],
        [0, 0, 1, 1, 0, 0],
        [0

In [113]:
variables[2,:]

array([13, 14, 15, 16, 17, 18])

In [114]:
clause_tensor = np.multiply(connectivity_tensor,  variables[2,:])
clause_tensor

array([[[ 0,  0,  0,  0,  0,  0],
        [ 0,  0, 15, 16,  0,  0],
        [ 0, 14,  0, 16,  0, 18],
        [ 0, 14, 15,  0,  0, 18],
        [ 0,  0,  0,  0,  0,  0],
        [ 0,  0, 15, 16,  0,  0]],

       [[ 0,  0, 15, 16,  0,  0],
        [ 0,  0,  0,  0,  0,  0],
        [13,  0,  0, 16,  0,  0],
        [13,  0, 15,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0]],

       [[ 0, 14,  0, 16,  0, 18],
        [13,  0,  0, 16,  0,  0],
        [ 0,  0,  0,  0,  0,  0],
        [13, 14,  0,  0, 17, 18],
        [ 0,  0,  0, 16,  0, 18],
        [13,  0,  0, 16, 17,  0]],

       [[ 0, 14, 15,  0,  0, 18],
        [13,  0, 15,  0,  0,  0],
        [13, 14,  0,  0, 17, 18],
        [ 0,  0,  0,  0,  0,  0],
        [ 0,  0, 15,  0,  0, 18],
        [13,  0, 15,  0, 17,  0]],

       [[ 0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0],
        [ 0,  0,  0, 16,  0, 18],
        [ 0,  0, 15,  0,  0, 18],
        [ 0,  0,  0,  0,  0,  0],
      

In [118]:
clause = np.concatenate(
    (
        [[[-variables[0, i], -variables[1,j]] for j in range(num_nodes_g2)]  for i in range(num_nodes_g2)],
        clause_tensor,
    ),
    axis=2,
)

In [116]:
for cc in clause:
    for c in cc:
        print(c[c != 0])

[-1 -7]
[-1 -8 15 16]
[-1 -9 14 16 18]
[ -1 -10  14  15  18]
[ -1 -11]
[ -1 -12  15  16]
[-2 -7 15 16]
[-2 -8]
[-2 -9 13 16]
[ -2 -10  13  15]
[ -2 -11]
[ -2 -12]
[-3 -7 14 16 18]
[-3 -8 13 16]
[-3 -9]
[ -3 -10  13  14  17  18]
[ -3 -11  16  18]
[ -3 -12  13  16  17]
[-4 -7 14 15 18]
[-4 -8 13 15]
[-4 -9 13 14 17 18]
[ -4 -10]
[ -4 -11  15  18]
[ -4 -12  13  15  17]
[-5 -7]
[-5 -8]
[-5 -9 16 18]
[ -5 -10  15  18]
[ -5 -11]
[ -5 -12  15  16]
[-6 -7 15 16]
[-6 -8]
[-6 -9 13 16 17]
[ -6 -10  13  15  17]
[ -6 -11  15  16]
[ -6 -12]


In [152]:
with Solver(bootstrap_with=cnf, use_timer=False) as solver:
    status = solver.solve_limited(expect_interrupt=False)
    sol = solver.get_model()

In [153]:
status

True

In [154]:
sol

[-1,
 -2,
 -3,
 -4,
 -5,
 6,
 -7,
 -8,
 -9,
 -10,
 11,
 -12,
 -13,
 -14,
 -15,
 16,
 -17,
 -18,
 -19,
 -20,
 21,
 -22,
 -23,
 -24,
 -25,
 26,
 -27,
 -28,
 -29,
 -30,
 31,
 -32,
 -33,
 -34,
 -35,
 -36]

In [155]:
mapping = [vid2mapping[idx] for idx in sol if idx > 0]

In [156]:
mapping

[(0, 5), (1, 4), (2, 3), (3, 2), (4, 1), (5, 0)]

In [157]:
edge_map = dict(mapping)
edge_map

{0: 5, 1: 4, 2: 3, 3: 2, 4: 1, 5: 0}

In [160]:
hamiltonian.num_qubits

12

In [162]:
print(hamiltonian)

SparsePauliOp(['IIIIIIIIIIII', 'IIIIIIIIIIIZ', 'IIIIIIIIIIZI', 'IIIIIIIIZIII', 'IIIIIIIZIIII', 'IIIIIZIIIIII', 'IIIIZIIIIIII', 'IIZIIIIIIIII', 'IZIIIIIIIIII', 'IIIIIIIIIIZZ', 'IIIIIIIIZIIZ', 'IIIIIIIIZIZI', 'IIIIIIIZIIIZ', 'IIIIIIIZIIZI', 'IIIIIIIZIZII', 'IIIIIIIZZIII', 'IIIIIIZIIIZI', 'IIIIIIZIIZII', 'IIIIIZIIIIIZ', 'IIIIIZIIIIZI', 'IIIIIZIIZIII', 'IIIIIZIZIIII', 'IIIIZIIIIIIZ', 'IIIIZIIIIIZI', 'IIIIZIIIZIII', 'IIIIZIIZIIII', 'IIIIZIZIIIII', 'IIIIZZIIIIII', 'IIIZIIIZIIII', 'IIIZIIZIIIII', 'IIZIIIIIIIIZ', 'IIZIIIIIIIZI', 'IIZIIIIIZIII', 'IIZIIIIZIIII', 'IIZIIZIIIIII', 'IIZIZIIIIIII', 'IZIIIIIIIIIZ', 'IZIIIIIIIIZI', 'IZIIIIIIZIII', 'IZIIIIIZIIII', 'IZIIIZIIIIII', 'IZIIZIIIIIII', 'IZIZIIIIIIII', 'IZZIIIIIIIII', 'ZIIIZIIIIIII', 'ZIIZIIIIIIII', 'IIIIIIIIZIZZ', 'IIIIIIIIZZZI', 'IIIIIIIZIIZZ', 'IIIIIIIZIZZI', 'IIIIIIIZZIIZ', 'IIIIIIIZZIZI', 'IIIIIIIZZZII', 'IIIIIIZIIIZZ', 'IIIIIIZIIZIZ', 'IIIIIIZIIZZI', 'IIIIIIZIZZII', 'IIIIIIZZIIIZ', 'IIIIIIZZIIZI', 'IIIIIIZZIZII', 'IIIIIZIIIIZZ', 'IIIIIZIZ

In [172]:
h = SparsePauliOp('ZZIIII', 1)
print(h)

SparsePauliOp(['ZZIIII'],
              coeffs=[1.+0.j])


In [170]:
edge_map = dict(mapping)

new_h = h.apply_layout([edge_map[i] for i in range(6)], 6)

In [171]:
print(new_h)

SparsePauliOp(['IIIIZZ'],
              coeffs=[1.+0.j])


In [174]:
len(hamiltonian)

151